# Modellek összehasonlítása

**Felelős:** Kovács Kornél

Ez a notebook beolvassa a `results/metrics.csv`-t (amibe minden modell-tanító notebook írta a saját eredményét), és vizuálisan + táblázatosan összehasonlítja őket. Saját maga semmit nem tanít, csak aggregál — ezért a többi után fut.

**Előfeltétel:** `02_baselines.ipynb`, `03_decision_tree.ipynb`, `05_hgbr.ipynb` lefutottak és írtak a `metrics.csv`-be.

Mentett kimenetek (Drive: `FIGURES_DIR`):
- `comparison_mae.png` — MAE bar chart minden modellre
- `comparison_r2.png` — R² bar chart minden modellre

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Slityak/gepitan-seoul-bike-trip/blob/main/notebooks/04_comparison.ipynb)

## 1. Setup (Colab + lokális kompatibilis)

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'main'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.exists('/content/gepitan-beadando'):
        !git clone --branch {BRANCH} https://github.com/Slityak/gepitan-seoul-bike-trip.git /content/gepitan-beadando

    os.chdir('/content/gepitan-beadando')
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
    !pip install -q -r requirements.txt
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')
print(f'In Colab: {IN_COLAB}')
print(f'Branch: {BRANCH}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.config import FIGURES_DIR, ensure_dirs
from src.evaluation import load_metrics
from src.visualization import plot_model_comparison

ensure_dirs()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 2. Metrikák betöltése

In [ ]:
metrics_df = load_metrics()
metrics_df

## 3. Összehasonlító táblázat

In [ ]:
report_df = metrics_df[['model_name', 'mae', 'rmse', 'r2', 'train_time_sec', 'predict_time_sec']].copy()
report_df = report_df.sort_values('mae')
report_df.round(3)

## 4. Vizuális összehasonlítás

In [ ]:
plot_model_comparison(metrics_df, metric='mae', save_path='comparison_mae.png')
plt.show()

In [ ]:
plot_model_comparison(metrics_df, metric='r2', save_path='comparison_r2.png')
plt.show()

## 5. Tanulságok

**Final ranking (test MAE szerint):**

| modell | MAE | RMSE | R² |
|---|---|---|---|
| Mean baseline | 19.19 | 25.04 | 0.00 |
| Median baseline | 17.56 | 26.90 | -0.15 |
| Linear Regression (Ridge) | 13.24 | 20.07 | 0.36 |
| Decision Tree (default) | 12.40 | 22.89 | 0.16 |
| Decision Tree (tuned) | 12.45 | 19.35 | 0.40 |
| HGBR (default) | 12.39 | 19.22 | 0.41 |
| **HGBR (tuned + log-target)** | **10.91** | 19.62 | 0.39 |

### Mit látunk

**A baseline-okhoz képest minden modell ér valamit.** A median-baseline (17.56 MAE) jelentősen jobb a mean-nál (19.19), mert a Duration eloszlása jobbra ferde — ez azonnal igazolja, hogy az MAE-orientált megközelítés helyes a feladathoz. A median R²-je negatív (-0.15), de ez nem hiba: a `r2_score` a mean-hez viszonyít, és egy ferde eloszlás medianja per definíció elhagyja a meant.

**A Ridge regresszió a feature-szet alapértékét adja meg.** MAE 13.24, R² 0.36 — a 36 feature lineárisan ennyi információt tartalmaz a Duration-ról. Minden ezen felüli javulás a *nem-lineáris* struktúrából származik.

**A Decision Tree default vs tuned különbsége tankönyvi.** A default fa MAE-ben hasonló (12.40), de R² csak 0.16, RMSE/MAE arány 1.85 → a `max_depth=None` overfittel és néhány hosszú útnál drámaian téved. A hangolás (max_depth=15, min_samples_leaf=100) ezt rendbe teszi: R² 0.40-re ugrik, RMSE 22.89→19.35, az arány 1.55-re javul. A MAE alig változik — a tuning a *robusztusságot* hozta meg, nem a tipikus hibát.

**A HGBR default már a tuned DT szintjén van.** MAE 12.39, R² 0.41 — ez azt jelenti, hogy a gradient boosting belső regularizációja (sok kis lépés) magától stabilizálja a modellt, hangolás nélkül is. A validation curve ezt megerősíti: a `max_leaf_nodes` 7 és 127 közötti változtatása mindössze 0.085 MAE különbséget okoz, szemben a DT U-alakú görbéjével.

**A HGBR tuned + log-target adja a legjobb MAE-t (10.91), de érdekes trade-off-fal.** A log1p/expm1 wrapper a tipikus utakon javít (MAE -12% a tuned DT-hez képest), de a hosszú farokra rosszabbul illeszkedik (RMSE 19.62 > a default HGBR 19.22, R² 0.39 < 0.41). Ez nem hiba, hanem a log-target jellegzetes ujjlenyomata: a square error nagyobb büntetést kap a long tail-en, ahol az `expm1` exponenciálisan nagyítja a kis log-térbeli hibákat.

### Modellválasztás

Ha a feladat MAE-orientált (a tipikus út pontos becslése), a **HGBR tuned + log-target a végső modell** (MAE 10.91, R² 0.39). Ha a varianciát kell maximálisan magyarázni, a **HGBR default** ad jobb R²-t (0.41) marginális MAE-veszteség mellett (12.39 vs 10.91).

### A learning curve-ök üzenete

A két modell adat-igénye fundamentálisan eltér:
- A **Decision Tree** 25k-30k mintánál plateauzik (CV MAE ~13.0). Akármennyi adatot adunk neki, ennél jobbat nem produkál — *kapacitás-korlátos*.
- A **HGBR** 40k-nál még mindig javul (CV MAE 12.57, lefelé tartó trend), és a teljes 7.68M-on érte el a 10.91 test MAE-t — *adat-éhes*.

Ez magyarázza, miért váltottunk ensemble-re egy egy-fa helyett: a fa-modell elérte a saját plafonját, a boosting viszont tovább skálázódik az adattal.

### Trade-off: tanítási idő

A modellek `train_time_sec` oszlopa megmutatja, hogy a teljesítmény-ugrás ára számottevő:
- baseline-ok: < 0.1 s
- Ridge: ~4 s
- Decision Tree (tuned): ~9 perc (a 9.6M soros refit dominál)
- HGBR (tuned + log-target): ~30-60 perc (700 boosting iteráció a teljes train-en)

A HGBR ~6× lassabb mint a DT, cserébe 12% jobb MAE-t és összehasonlítható R²-t ad.